In [1]:
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [2]:
FINAL_DIR = PROJECT_ROOT / "data" / "final"
IMAGE_INPUT_PATH = FINAL_DIR / "final_cues_for_image_generation.csv"

RAW_IMAGE_DIR = PROJECT_ROOT / "results" / "generated_images_raw"
MOBILE_IMAGE_DIR = PROJECT_ROOT / "results" / "generated_images_mobile"

RAW_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
MOBILE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("IMAGE INPUT PATH:", IMAGE_INPUT_PATH)
print("RAW IMAGE DIR:", RAW_IMAGE_DIR)
print("MOBILE IMAGE DIR:", MOBILE_IMAGE_DIR)

IMAGE INPUT PATH: c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\final_cues_for_image_generation.csv
RAW IMAGE DIR: c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\results\generated_images_raw
MOBILE IMAGE DIR: c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\results\generated_images_mobile


In [3]:
image_generation_df = pd.read_csv(IMAGE_INPUT_PATH)

print("IMAGE GENERATION DF SHAPE:", image_generation_df.shape)
display(image_generation_df.head(20))

IMAGE GENERATION DF SHAPE: (125, 7)


,target_word,subcategory,best_prompt,reference_best_human_cue,final_generated_cue,proxy_score,image_prompt
0,火車,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,"需要購票搭乘,有分自由座和對號座",需要購票才能登車,0.881714,"a realistic 火車, clear full view, simple backgr..."
1,汽車,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,需要加油才能行駛,需要車道才能行駛,0.705243,"a realistic 汽車, clear full view, simple backgr..."
2,自行車,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,運動或代步皆適合,騎行方便，速度快，適合短途出行,0.684109,"a realistic 自行車, clear full view, simple backg..."
3,高鐵,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,"透過電力驅動,行駛於專用軌道上",需要刷卡才能進站,0.511423,"a realistic 高鐵, clear full view, simple backgr..."
4,公車,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,票價便宜,上車後，找到自己的座位，系好安全帶,0.498135,"a realistic 公車, clear full view, simple backgr..."
5,摩托車,交通工具,你是一位專門為高齡失語症患者設計語言治療提示句的專家。你的任務是接收一個目標交通工具詞彙，並...,需要加汽油或電力才能運行,騎行在城市街道上，感受風吹拂臉頰,0.421682,"a realistic 摩托車, clear full view, simple backg..."
6,泡茶,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,選擇合適的茶具與水質,將水倒入茶壺，加入紅茶包和糖，輕輕,0.863611,a realistic illustration of the action or acti...
7,下象棋,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,需要理解不同棋子的移動規則才能下得好,盤上落子，棋局起始，靜候勝負爭奪,0.672283,a realistic illustration of the action or acti...
8,跳舞,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,與拍手、踩腳或轉圈的動作結合,在空曠場地隨意扭轉身體，與同伴互動，,0.655932,a realistic illustration of the action or acti...
9,唱歌,休閒娛樂,你是一位專門為高齡失語症患者設計語言治療提示句的專家。請根據目標詞彙最具辨識度的特徵，生成一...,"有時會搭配樂器,如吉他、鋼琴或卡拉OK伴奏",在空蕩的房間裡唱著歌，讓心情更加愉快,0.598839,a realistic illustration of the action or acti...


In [4]:
SDXL_LOCAL_MODEL_PATH = r"C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\models\sdxl-base-1.0"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("DEVICE:", device)
print("DTYPE:", dtype)
print("MODEL PATH:", SDXL_LOCAL_MODEL_PATH)

pipe = StableDiffusionXLPipeline.from_pretrained(
    SDXL_LOCAL_MODEL_PATH,
    torch_dtype=dtype,
    local_files_only=True,
    use_safetensors=True,
)

if device == "cuda":
    pipe = pipe.to("cuda")
else:
    pipe.enable_model_cpu_offload()

print("SDXL pipeline loaded successfully.")

DEVICE: cuda
DTYPE: torch.float16
MODEL PATH: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\models\sdxl-base-1.0


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

SDXL pipeline loaded successfully.


In [5]:
def safe_filename(text: str) -> str:
    text = str(text).strip()
    bad_chars = ['\\', '/', ':', '*', '?', '"', '<', '>', '|']
    for ch in bad_chars:
        text = text.replace(ch, "_")
    return text


def build_negative_prompt(subcategory: str) -> str:
    return (
        "blurry, low quality, distorted, extra objects, cluttered background, "
        "text, watermark, logo, cropped, duplicate, messy scene"
    )


def generate_raw_image(
    prompt: str,
    negative_prompt: str,
    output_path: Path,
    seed: int = 42,
    width: int = 768,
    height: int = 768,
    num_inference_steps: int = 30,
    guidance_scale: float = 7.0,
):
    generator = torch.Generator(device=device).manual_seed(seed)

    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    ).images[0]

    image.save(output_path)
    return output_path


def export_mobile_image(raw_image_path: Path, mobile_image_path: Path, size=(512, 512), quality=90):
    img = Image.open(raw_image_path).convert("RGB")
    img = img.resize(size, Image.LANCZOS)
    img.save(mobile_image_path, format="JPEG", quality=quality, optimize=True)
    return mobile_image_path

In [6]:
test_df = image_generation_df.head(5).copy()
display(test_df[["target_word", "subcategory", "final_generated_cue", "image_prompt"]])

,target_word,subcategory,final_generated_cue,image_prompt
0,火車,交通工具,需要購票才能登車,"a realistic 火車, clear full view, simple backgr..."
1,汽車,交通工具,需要車道才能行駛,"a realistic 汽車, clear full view, simple backgr..."
2,自行車,交通工具,騎行方便，速度快，適合短途出行,"a realistic 自行車, clear full view, simple backg..."
3,高鐵,交通工具,需要刷卡才能進站,"a realistic 高鐵, clear full view, simple backgr..."
4,公車,交通工具,上車後，找到自己的座位，系好安全帶,"a realistic 公車, clear full view, simple backgr..."


In [7]:
test_rows = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating TEST images"):
    target_word = row["target_word"]
    subcategory = row["subcategory"]
    final_generated_cue = row["final_generated_cue"]
    image_prompt = row["image_prompt"]

    raw_subcat_dir = RAW_IMAGE_DIR / safe_filename(subcategory)
    mobile_subcat_dir = MOBILE_IMAGE_DIR / safe_filename(subcategory)

    raw_subcat_dir.mkdir(parents=True, exist_ok=True)
    mobile_subcat_dir.mkdir(parents=True, exist_ok=True)

    raw_image_path = raw_subcat_dir / f"{safe_filename(target_word)}.png"
    mobile_image_path = mobile_subcat_dir / f"{safe_filename(target_word)}.jpg"

    generation_status = "pending"
    error_message = ""

    try:
        negative_prompt = build_negative_prompt(subcategory)

        generate_raw_image(
            prompt=image_prompt,
            negative_prompt=negative_prompt,
            output_path=raw_image_path,
            seed=42,
            width=768,
            height=768,
            num_inference_steps=30,
            guidance_scale=7.0,
        )

        export_mobile_image(
            raw_image_path=raw_image_path,
            mobile_image_path=mobile_image_path,
            size=(512, 512),
            quality=90,
        )

        generation_status = "success"

    except Exception as e:
        generation_status = "failed"
        error_message = str(e)

    test_rows.append({
        "target_word": target_word,
        "subcategory": subcategory,
        "final_generated_cue": final_generated_cue,
        "image_prompt": image_prompt,
        "raw_image_path": str(raw_image_path),
        "mobile_image_path": str(mobile_image_path),
        "generation_status": generation_status,
        "error_message": error_message,
    })

test_results_df = pd.DataFrame(test_rows)

print("TEST RESULTS SHAPE:", test_results_df.shape)
display(test_results_df)

Generating TEST images:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

c:\Users\804\anaconda3\envs\aphasia_ape\lib\site-packages\diffusers\pipelines\stable_diffusion_xl\pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
Generating TEST images:  20%|██        | 1/5 [00:13<00:55, 13.86s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

Generating TEST images:  40%|████      | 2/5 [00:26<00:39, 13.32s/it]

...

Generating TEST images:  80%|████████  | 4/5 [00:52<00:13, 13.02s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

Generating TEST images: 100%|██████████| 5/5 [01:05<00:00, 13.09s/it]

TEST RESULTS SHAPE: (5, 8)


,target_word,subcategory,final_generated_cue,image_prompt,raw_image_path,mobile_image_path,generation_status,error_message
0,火車,交通工具,需要購票才能登車,"a realistic 火車, clear full view, simple backgr...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
1,汽車,交通工具,需要車道才能行駛,"a realistic 汽車, clear full view, simple backgr...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
2,自行車,交通工具,騎行方便，速度快，適合短途出行,"a realistic 自行車, clear full view, simple backg...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
3,高鐵,交通工具,需要刷卡才能進站,"a realistic 高鐵, clear full view, simple backgr...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
4,公車,交通工具,上車後，找到自己的座位，系好安全帶,"a realistic 公車, clear full view, simple backgr...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,


In [8]:
print("Successful test images:", (test_results_df["generation_status"] == "success").sum())
print("Failed test images:", (test_results_df["generation_status"] == "failed").sum())

Successful test images: 5
Failed test images: 0


In [9]:
image_rows = []

for _, row in tqdm(image_generation_df.iterrows(), total=len(image_generation_df), desc="Generating FULL images"):
    target_word = row["target_word"]
    subcategory = row["subcategory"]
    final_generated_cue = row["final_generated_cue"]
    image_prompt = row["image_prompt"]

    raw_subcat_dir = RAW_IMAGE_DIR / safe_filename(subcategory)
    mobile_subcat_dir = MOBILE_IMAGE_DIR / safe_filename(subcategory)

    raw_subcat_dir.mkdir(parents=True, exist_ok=True)
    mobile_subcat_dir.mkdir(parents=True, exist_ok=True)

    raw_image_path = raw_subcat_dir / f"{safe_filename(target_word)}.png"
    mobile_image_path = mobile_subcat_dir / f"{safe_filename(target_word)}.jpg"

    generation_status = "pending"
    error_message = ""

    try:
        negative_prompt = build_negative_prompt(subcategory)

        generate_raw_image(
            prompt=image_prompt,
            negative_prompt=negative_prompt,
            output_path=raw_image_path,
            seed=42,
            width=768,
            height=768,
            num_inference_steps=30,
            guidance_scale=7.0,
        )

        export_mobile_image(
            raw_image_path=raw_image_path,
            mobile_image_path=mobile_image_path,
            size=(512, 512),
            quality=90,
        )

        generation_status = "success"

    except Exception as e:
        generation_status = "failed"
        error_message = str(e)

    image_rows.append({
        "target_word": target_word,
        "subcategory": subcategory,
        "final_generated_cue": final_generated_cue,
        "image_prompt": image_prompt,
        "raw_image_path": str(raw_image_path),
        "mobile_image_path": str(mobile_image_path),
        "generation_status": generation_status,
        "error_message": error_message,
    })

image_results_df = pd.DataFrame(image_rows)

print("IMAGE RESULTS SHAPE:", image_results_df.shape)
display(image_results_df.head(20))

Generating FULL images:   0%|          | 0/125 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

c:\Users\804\anaconda3\envs\aphasia_ape\lib\site-packages\diffusers\pipelines\stable_diffusion_xl\pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
Generating FULL images:   1%|          | 1/125 [00:13<27:04, 13.10s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

Generating FULL images:   2%|▏         | 2/125 [00:25<26:35, 12.97s/it]

...

Generating FULL images:  99%|█████████▉| 124/125 [26:53<00:13, 13.05s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

Generating FULL images: 100%|██████████| 125/125 [27:06<00:00, 13.02s/it]

IMAGE RESULTS SHAPE: (125, 8)


,target_word,subcategory,final_generated_cue,image_prompt,raw_image_path,mobile_image_path,generation_status,error_message
0,火車,交通工具,需要購票才能登車,"a realistic 火車, clear full view, simple backgr...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
1,汽車,交通工具,需要車道才能行駛,"a realistic 汽車, clear full view, simple backgr...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
2,自行車,交通工具,騎行方便，速度快，適合短途出行,"a realistic 自行車, clear full view, simple backg...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
3,高鐵,交通工具,需要刷卡才能進站,"a realistic 高鐵, clear full view, simple backgr...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
4,公車,交通工具,上車後，找到自己的座位，系好安全帶,"a realistic 公車, clear full view, simple backgr...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
5,摩托車,交通工具,騎行在城市街道上，感受風吹拂臉頰,"a realistic 摩托車, clear full view, simple backg...",c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
6,泡茶,休閒娛樂,將水倒入茶壺，加入紅茶包和糖，輕輕,a realistic illustration of the action or acti...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
7,下象棋,休閒娛樂,盤上落子，棋局起始，靜候勝負爭奪,a realistic illustration of the action or acti...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
8,跳舞,休閒娛樂,在空曠場地隨意扭轉身體，與同伴互動，,a realistic illustration of the action or acti...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,
9,唱歌,休閒娛樂,在空蕩的房間裡唱著歌，讓心情更加愉快,a realistic illustration of the action or acti...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...,success,


In [10]:
generated_image_results_path = FINAL_DIR / "generated_image_results.csv"
image_results_df.to_csv(generated_image_results_path, index=False, encoding="utf-8-sig")

print("Saved:", generated_image_results_path)

Saved: c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\generated_image_results.csv


In [11]:
mobile_export_df = image_results_df[image_results_df["generation_status"] == "success"].copy()

mobile_export_df = mobile_export_df[[
    "target_word",
    "subcategory",
    "final_generated_cue",
    "mobile_image_path",
]].copy()

mobile_export_df = mobile_export_df.rename(columns={
    "mobile_image_path": "image_path"
})

mobile_export_path = FINAL_DIR / "mobile_ready_cue_image_package.csv"
mobile_export_df.to_csv(mobile_export_path, index=False, encoding="utf-8-sig")

print("Saved:", mobile_export_path)
display(mobile_export_df.head(20))

Saved: c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\final\mobile_ready_cue_image_package.csv


,target_word,subcategory,final_generated_cue,image_path
0,火車,交通工具,需要購票才能登車,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
1,汽車,交通工具,需要車道才能行駛,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
2,自行車,交通工具,騎行方便，速度快，適合短途出行,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
3,高鐵,交通工具,需要刷卡才能進站,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
4,公車,交通工具,上車後，找到自己的座位，系好安全帶,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
5,摩托車,交通工具,騎行在城市街道上，感受風吹拂臉頰,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
6,泡茶,休閒娛樂,將水倒入茶壺，加入紅茶包和糖，輕輕,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
7,下象棋,休閒娛樂,盤上落子，棋局起始，靜候勝負爭奪,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
8,跳舞,休閒娛樂,在空曠場地隨意扭轉身體，與同伴互動，,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...
9,唱歌,休閒娛樂,在空蕩的房間裡唱著歌，讓心情更加愉快,c:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Gene...


In [12]:
success_df = image_results_df[image_results_df["generation_status"] == "success"].copy()
failed_df = image_results_df[image_results_df["generation_status"] == "failed"].copy()

print("Successful images:", len(success_df))
print("Failed images:", len(failed_df))

if len(failed_df) > 0:
    display(failed_df.head(20))

Successful images: 125
Failed images: 0
